# Aula 2 – Vídeo 2: Aplicação Simples com LangGraph

Neste notebook vamos criar uma aplicação mínima com dois nós:
- Um nó LLM que resume um texto.
- Um nó Python que conta as palavras do resumo.


In [1]:
import os
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import PromptTemplate
except Exception:
    ChatOpenAI = None
    PromptTemplate = None

load_dotenv()
llm = None
if ChatOpenAI:
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    llm = ChatOpenAI(model=model, temperature=0)

# Estado tipado
class State(TypedDict, total=False):
    texto: str
    resumo: str
    num_palavras: int

def resumir(s: State) -> State:
    if llm and PromptTemplate:
        prompt = PromptTemplate.from_template("Resuma em uma frase: {texto}")
        chain = prompt | llm
        resumo = chain.invoke({"texto": s["texto"]}).content
        return {"resumo": resumo}
    return {"resumo": s["texto"][:40] + "..."}

def contar_palavras(s: State) -> State:
    resumo = s.get("resumo", "")
    return {"num_palavras": len(resumo.split())}

grafo = StateGraph(State)
grafo.add_node("resumir", resumir)
grafo.add_node("contar", contar_palavras)
grafo.set_entry_point("resumir")
grafo.add_edge("resumir", "contar")
grafo.add_edge("contar", END)
app = grafo.compile()

estado_inicial = {"texto": "LangGraph organiza fluxos de IA como grafos de execução."}
print(app.get_graph().draw_ascii())
print(app.invoke(estado_inicial))


+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | resumir |   
 +---------+   
      *        
      *        
      *        
  +--------+   
  | contar |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   
{'texto': 'LangGraph organiza fluxos de IA como grafos de execução.', 'resumo': 'LangGraph estrutura fluxos de inteligência artificial em forma de grafos de execução.', 'num_palavras': 12}
